# Healthcare FHIR Integration : Évaluation du triage LLM

**Entrées** : 26 tickets de support **fictifs** étiquetés (`ai/support_tickets.json`)  
**LLM** : Llama 3.2 3B en local via Ollama, température 0  
**Prérequis** : Ollama lancé avec `llama3.2` (le test des 26 tickets prend environ une minute)  
**Stack** : Python, intervalle de Wilson à 95 % (`ai/evaluate_triage.py`)

---

### Objectif

Un LLM local peut classer des demandes de support d'interopérabilité (erreur de mapping, donnée manquante, problème de connexion...), mais sa réponse ne suffit pas. Ce notebook montre le LLM à l'œuvre, puis le teste sur 26 tickets, en séparant clairement la classification générative des contrôles déterministes faits par Python.

### Fonctionnement

1. **Deux usages du LLM** : expliquer une erreur HL7 et classer une demande de support.
2. **Catégories imposées** : pour classer une demande, le LLM doit choisir l'une des 5 catégories prévues (erreur de mapping, donnée manquante, problème de connexion, question de format, autre) ; toute autre réponse est rejetée par Python.
3. **Configuration du LLM** : modèle, température, prompts.
4. **Test en direct** sur 26 tickets étiquetés : bonnes réponses, réponses rejetées, erreurs, avec un intervalle de confiance (Wilson) car 26 tickets font un petit échantillon.

### Limite assumée

Les tickets et le prompt ont été écrits par la même personne sur un très petit jeu : le score est optimiste et illustre une méthode d'évaluation, pas une performance en conditions réelles.

## 1. Chargement des tickets

Les 26 tickets sont des demandes de support **fictives**, rédigées pour ce projet (`ai/support_tickets.json`), sans aucune donnée réelle. Chacune est étiquetée à la main avec la catégorie attendue : ce sont ces étiquettes qui permettent de compter les bonnes réponses.


In [ ]:
from collections import Counter
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

tickets = json.loads((PROJECT_ROOT / "ai" / "support_tickets.json").read_text(encoding="utf-8"))

print(f"Tickets fictifs : {len(tickets)}")
for category, count in Counter(ticket["expected"] for ticket in tickets).items():
    print(f"- {category} : {count}")

Tickets fictifs : 26
- erreur_de_mapping : 5
- donnee_manquante : 6
- probleme_de_connexion : 5
- question_de_format : 5
- autre : 5


> **Résultat.** 26 tickets sont chargés, répartis en 5 catégories (5 tickets par catégorie, sauf 6 pour `donnee_manquante`).

## 2. Décision d’architecture

Pour classer une demande de support, le modèle choisit uniquement l'une des 5 catégories prévues (affichées ci-dessous). Python rejette toute catégorie inconnue, conserve les décisions critiques (parsing, validation et sévérité) et permet aux pipelines FHIR/HL7 de fonctionner lorsque Ollama est indisponible.


In [3]:
from ai.ticket_triage import CATEGORIES

print(f"Nombre de catégories autorisées : {len(CATEGORIES)}")
for name, description in CATEGORIES.items():
    print(f"- {name}: {description}")


Nombre de catégories autorisées : 5
- erreur_de_mapping: une valeur est mal convertie ou associée au mauvais champ entre deux formats
- donnee_manquante: un champ ou un enregistrement est vide, absent ou incomplet
- probleme_de_connexion: les échanges avec un système sont coupés, lents ou en erreur (timeout, 500...)
- question_de_format: question sur la structure d'un standard (HL7, FHIR, JSON) ou sur sa documentation
- autre: ne correspond à aucune autre catégorie


> **Décision.** Le LLM reste une aide facultative sur une tâche à faible criticité. Une sortie conforme au JSON n’est pas nécessairement vraie : la validation garantit le contrat syntaxique, pas la qualité sémantique de la catégorie.

## 3. Configuration du LLM

Les deux usages du LLM (diagnostic d'erreur et tri des demandes) partagent le même réglage, défini dans `ai/interop_assistant.py` :

| Paramètre | Valeur | Pourquoi |
| :--- | :--- | :--- |
| **Modèle** | Llama 3.2 3B (`llama3.2`), exécuté en local avec Ollama | Les données restent sur la machine et il n'y a aucun coût d'API. |
| **Température** | 0 | Le modèle choisit toujours la réponse la plus probable : la même demande donne la même réponse, ce qui rend les mesures reproductibles. |
| **Format de sortie** | JSON imposé (`"format": "json"`) | La réponse est lisible par le code, pas seulement par un humain. |
| **Délai maximum** | 60 secondes | Une panne d'Ollama ne bloque pas le pipeline. |
| **Exemples donnés au modèle** | Aucun | Le modèle reçoit seulement des consignes, sans exemple de réponse attendue. |

À chaque appel, le modèle reçoit deux messages : un **prompt système**, qui fixe son rôle et ses règles, et un **prompt utilisateur**, qui contient la tâche et les données du cas. Ce sont les textes exacts utilisés par le code :


In [ ]:
from ai.interop_assistant import SYSTEM_PROMPT, build_error_context, build_user_prompt
from ai.ticket_triage import TRIAGE_SYSTEM_PROMPT, build_triage_prompt

print("=== Diagnostic d'erreur : prompt système ===")
print(SYSTEM_PROMPT.strip())

print("\n=== Diagnostic d'erreur : prompt utilisateur (les valeurs entre chevrons sont remplacées à chaque appel) ===")
print(build_user_prompt(build_error_context("<erreur détectée>", "<message HL7>", "<sévérité>")).strip())

print("\n=== Tri des demandes : prompt système ===")
print(TRIAGE_SYSTEM_PROMPT.strip())

print("\n=== Tri des demandes : prompt utilisateur ===")
print(build_triage_prompt("<texte du ticket>").strip())

=== Diagnostic d'erreur : prompt système ===
Tu es un assistant technique spécialisé en interopérabilité des systèmes d'information en santé.

Tu analyses des erreurs provenant d'un pipeline HL7 v2 vers FHIR.

Règles :
- N'invente jamais de donnée patient.
- Ne complète jamais une donnée HL7 absente.
- Ne modifie jamais le message HL7 source.
- Analyse uniquement les informations techniques fournies.
- Si une information est inconnue, indique qu'elle est inconnue.
- Propose uniquement des actions techniques de diagnostic ou de correction.

Ta réponse doit identifier :
- le type d'erreur ;
- l'élément HL7 concerné ;
- l'impact sur la ressource FHIR ;
- une explication technique ;
- une action technique suggérée.

=== Diagnostic d'erreur : prompt utilisateur (les valeurs entre chevrons sont remplacées à chaque appel) ===
Analyse l'erreur d'interopérabilité suivante.

Erreur détectée :
<erreur détectée>

Sévérité déterminée par le pipeline :
<sévérité>

Message HL7 :
<message HL7>

Retour

> **Résultat.** Dans les deux cas, le prompt interdit d'inventer des données et impose une réponse JSON. Pour le diagnostic, la sévérité est donnée par Python et le modèle doit la recopier. Pour le tri, le modèle choisit l'une des 5 catégories prévues. La réponse est ensuite vérifiée par Python (champs présents, sévérité imposée, catégorie autorisée) et rejetée si elle ne respecte pas ces règles.

## 4. Le LLM explique une erreur

Dans le notebook 02, on a converti un message HL7 en fiche patient FHIR. Un message peut aussi arriver incomplet, par exemple sans le segment `PID` qui contient l'identité du patient (le logiciel émetteur ne l'a pas envoyé) : la conversion est alors impossible. Python détecte le problème et le déclare bloquant. Le LLM sert ensuite à expliquer cette erreur simplement, pour quelqu'un qui ne lit pas du HL7.

On l'essaie sur le message du notebook 02 auquel on a retiré la ligne `PID`. Ollama doit être lancé.

In [ ]:
from ai.interop_assistant import diagnose_interop_error
from hl7.hl7_to_fhir import find_pid_segment

message_sans_pid = r"MSH|^~\&|HOSPITAL_A|PARIS|CONNECTOR|APP|202609182200||ADT^A01|MSG00001|P|2.5"

try:
    find_pid_segment(message_sans_pid)
except ValueError as error:
    print(f"Erreur détectée par Python : {error}")
    diagnostic = diagnose_interop_error(str(error), message_sans_pid, "blocking")

print(json.dumps(diagnostic, indent=2, ensure_ascii=False))

Erreur détectée par Python : Aucun segment PID trouvé
{
  "error_type": "Segment manquant",
  "severity": "blocking",
  "hl7_element": "PID",
  "fhir_impact": "Ressource FHIR non créée",
  "explanation": "Le segment PID est essentiel pour identifier le patient dans le message HL7.",
  "suggested_action": "Vérifier que le segment PID est correctement envoyé dans le message HL7."
}


> **Résultat.** Le diagnostic est structuré et cohérent avec l'erreur. Sa sévérité `blocking` vient de Python, pas du LLM : le code impose cette valeur et rejette tout diagnostic dont les champs sont invalides.

## 5. Test du tri sur les 26 tickets

On lance le LLM sur chaque ticket (`python -m ai.evaluate_triage` fait la même chose en ligne de commande). Pour chacun, il choisit une catégorie, que l'on compare à la catégorie attendue. Une réponse hors de la liste est rejetée par Python et comptée comme fausse. Ollama doit être lancé ; l'exécution prend environ une minute.


In [ ]:
from ai.evaluate_triage import evaluate, print_report

outcome = evaluate(tickets)
print_report(outcome, len(tickets))

Résultat : 23 bonnes réponses sur 26
Exactitude : 88.5% (IC 95 % de Wilson : 71.0%–96.0%)
Réponses rejetées par la validation : 0

- Le nom de famille se retrouve dans le champ prénom pour les patients du centre de santé.
  attendu : erreur_de_mapping | obtenu : donnee_manquante

- À quelle heure est la réunion de synchronisation de jeudi ?
  attendu : autre | obtenu : question_de_format

- Un patient n'apparaît pas chez nous, est-ce un problème de synchronisation ?
  attendu : donnee_manquante | obtenu : question_de_format


> **Résultat.** Le LLM classe correctement **23 tickets sur 26 (88,5 %)**, avec un intervalle de confiance à 95 % de Wilson d'environ 71,0 % à 96,0 %. Aucune réponse n'a été rejetée par la validation. Les 3 erreurs sont listées ci-dessus : un nom de famille dans le champ prénom classé `donnee_manquante` au lieu de `erreur_de_mapping`, une question sur l'heure d'une réunion classée `question_de_format` au lieu de `autre`, et un patient qui n'apparaît pas, classé `question_de_format` au lieu de `donnee_manquante` parce que la demande est formulée comme une question.

---

## Conclusion

Ce notebook montre où le LLM a sa place dans le projet, comment il est réglé, et ce qu'il vaut sur un petit jeu de tickets fictifs.

### Ce qu'on a fait

| Étape | Résultat |
| :--- | :--- |
| **Tickets** | 26 demandes de support fictives, étiquetées à la main : 5 par catégorie, sauf 6 pour `donnee_manquante`. |
| **Décision d'architecture** | Pour classer une demande, le LLM doit choisir l'une des 5 catégories prévues (`erreur_de_mapping`, `donnee_manquante`, `probleme_de_connexion`, `question_de_format` ou `autre`). Python rejette toute autre réponse et garde les décisions critiques (conversion, validation, sévérité). Le LLM reste facultatif. |
| **Configuration** | Llama 3.2 3B en local avec Ollama, température 0, réponse JSON imposée, aucun exemple donné. Deux prompts par usage : un prompt système (rôle et règles) et un prompt utilisateur (la tâche et les données). |
| **Diagnostic d'erreur** | Sur un message sans segment `PID`, le LLM rend un diagnostic structuré (« Segment manquant », `blocking`, « Ressource FHIR non créée », action suggérée). La sévérité est imposée par Python. |
| **Test du tri** | **23 tickets sur 26 (88,5 %)**, intervalle de confiance à 95 % de Wilson de 71,0 % à 96,0 %, aucune réponse rejetée par la validation. |

### Limites

- **Petit jeu, écrit par la même personne que le prompt** : le score est optimiste. Il illustre une méthode d'évaluation, pas une aptitude au déploiement clinique.
- **La validation vérifie la forme, pas la justesse** : un ticket « un patient n'apparaît pas chez nous » est classé `question_de_format`, une catégorie valide donc acceptée, alors qu'il relevait de `donnee_manquante`.
- **Variabilité non mesurée** : l'intervalle de confiance ne tient compte que du petit nombre de tickets, pas des différences possibles d'une exécution à l'autre.
- **Diagnostic d'erreur non évalué** : il est montré sur un seul exemple, sans mesure de sa qualité.

### Pour la suite

1. Agrandir le jeu d'évaluation avec des demandes ambiguës et anonymisées.
2. Mesurer la variabilité entre exécutions.
3. Garder un jeu de non-régression indépendant de la conception du prompt.
4. Ne traiter aucune donnée réelle avant une revue de sécurité, de conformité et de gouvernance.

### Bilan des trois notebooks

| Notebook | Ce qu'il montre |
| :--- | :--- |
| **01** | Lecture d'un Bundle FHIR, validation des patients, enregistrement SQLite sans doublon. |
| **02** | Conversion d'un message HL7 v2 en ressource FHIR `Patient`, avec les cas limites. |
| **03** | Un LLM local qui aide au tri et au diagnostic, sous le contrôle du code Python. |

L'intégration FHIR, le mapping HL7 et la persistance restent du code Python ordinaire. Le LLM est isolé, optionnel, et évalué sur des données fictives.